# Model-Logistic Regression


In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

# import the clean and preprocess functions
from preprocess import preprocess_data
# import constants
from features import numeric_cols, binary_cols, multi_category_cols, parent_child_groups, categorical_cols, identifier_cols

In [2]:
pd.set_option('display.max_columns', None)
pd.options.display.max_colwidth = 500
pd.options.display.max_rows = 100

## Data Preprocessing

I will follow the Kuppermann's method: build two separate models for the age<2 and age >= 2 groups. 

Since Logistic Regression cannot directly deal with missingness and multi-category features, these are my preprocessing procedures:

+ **Missing**:
  + Binary: add indicator columns and fill `NaN` with 0
  + Multi-category: add missing category

+ **One-hot encoding**:
  + To avoid multicollinearity, drop the first level
+ **Impute GCS**:
  + infer the missisng component of GCS scores

+ **Drop rows with missingvalues directly**:
  + Based on my previous analysis, we simply have very trivial rows containing missingness after encoding. To avoid introducing wrong assumption, I choose the conservative way.

In [3]:
%run clean.py

In [4]:
tbi_cleaned_new = pd.read_csv('../data/TBI_cleaned.csv')

In [5]:
train_lr_data, val_lr_data, test_lr_data = preprocess_data(
    df=tbi_cleaned_new,
    numeric_cols=numeric_cols,
    categorical_cols=categorical_cols,
    multi_category_cols=multi_category_cols,
    binary_cols=binary_cols,
    parent_child_groups=parent_child_groups,
    target_col='PosIntFinal',
    test_size=0.2,
    val_size=0.1,
    random_state=42,
    stratify_col=['PosIntFinal', 'AgeTwoPlus'],
    if_exclude_gcs_under_13= False, # include GCS 3-13 different from paper
    if_fill_missing_gcs=True,
    gcs_fill_strategy="leave", # leave missing 
    if_one_hot_encode=True,
    drop_first_cat_in_ohe=True,
    if_drop_na_rows=True, # after all the missing values are handled, drop any remaining rows with NA
    if_handle_parent_child_missing=True,
    if_handle_missing_for_categories=True,
    missing_category_label="missing",
    if_add_flag_missing_cols=True, # add flag columns to indicate missingness for each feature
    parent_missing_strategy="fill0",
    child_missing_when_parent_yes="missing_category",
    binary_missing_strategy="fill0",
    multi_missing_strategy="missing_category"
)

In [6]:
train_lr_data.shape, val_lr_data.shape, test_lr_data.shape

((30320, 290), (4331, 290), (8664, 290))

In [7]:
train_lr_data.isna().sum().sort_values(ascending=False).head(3), val_lr_data.isna().sum().sort_values(ascending=False).head(3), test_lr_data.isna().sum().sort_values(ascending=False).head(3)

(EmplType      12
 ClavTem_1      0
 IndSeiz_92     0
 dtype: int64,
 EmplType      3
 ClavTem_1     0
 IndSeiz_92    0
 dtype: int64,
 EmplType      3
 ClavTem_1     0
 IndSeiz_92    0
 dtype: int64)

The missingness in identifier columns makes no difference.

Overall, we have 30320 observations in train set. Comparing it with the number in CDR modeling, which is 29687, they are quite close to each other. 

The reason why we have more observations is that we don't exclude the GCS 3-13 group at this stage.

In [8]:
# split the data into age < 2 and age >= 2 
# during the splitting, we have already stratified by age group, so the distribution of the target variable should be similar in both age groups. Therefore, we can directly split the data based on the 'AgeTwoPlus' column without worrying about the target variable distribution.
train_lr_data_age_lt_2 = train_lr_data[train_lr_data['AgeTwoPlus'] == 1]
train_lr_data_age_gte_2 = train_lr_data[train_lr_data['AgeTwoPlus'] == 2]
val_lr_data_age_lt_2 = val_lr_data[val_lr_data['AgeTwoPlus'] == 1]
val_lr_data_age_gte_2 = val_lr_data[val_lr_data['AgeTwoPlus'] == 2]
test_lr_data_age_lt_2 = test_lr_data[test_lr_data['AgeTwoPlus'] == 1]
test_lr_data_age_gte_2 = test_lr_data[test_lr_data['AgeTwoPlus'] == 2]

In [9]:
target_col = 'PosIntFinal'

We have a bunch of categorical variables, so we should first do feature selection.

In [10]:
# these are the post-CT features
downstream_var = ([

    # --- ED management / utilization outcomes ---
    "Observed",          # observed in ED
    "EDDisposition",     # discharge / admit / OR etc.
    "CTDone",            # any head CT performed
    "EDCT",              # CT performed in ED
    'CTForm1',         

    # --- CT results ---
    "PosCT",             # traumatic finding on CT

    # Individual CT findings (radiology outcomes)
    "Finding1",   # cerebellar hemorrhage
    "Finding2",   # cerebral contusion
    "Finding3",   # cerebral edema
    "Finding4",   # intracerebral hemorrhage
    "Finding5",   # skull diastasis
    "Finding6",   # epidural hematoma
    "Finding7",   # extra-axial hematoma
    "Finding8",   # intraventricular hemorrhage
    "Finding9",   # midline shift
    "Finding10",  # pneumocephalus
    "Finding11",  # skull fracture
    "Finding12",  # subarachnoid hemorrhage
    "Finding13",  # subdural hematoma
    "Finding14",  # traumatic infarction
    "Finding20",  # diffuse injury (PI added)
    "Finding21",  # herniation
    "Finding22",  # shear injury
    "Finding23",  # sigmoid sinus injury

    # --- Clinical outcomes ---
    "DeathTBI",        # death due to TBI
    "HospHead",        # hospitalized ≥2 nights
    "HospHeadPosCT",   # hospitalization with positive CT
    "Intub24Head",     # intubated >24h
    "Neurosurgery",    # neurosurgical intervention

    # --- Primary study endpoint ---
    "PosIntFinal"      # clinically important TBI (ciTBI)
] + [col for col in train_lr_data.columns if col.find('Ind')!=-1] # CT assignment indicators
+ [col for col in train_lr_data.columns if col.find('CTSed')!=-1] # pharmacological sedation for head CT scan and related child indicators
+ [col for col in train_lr_data.columns if col.find('Finding')!=-1] # get rid of the related one-hot encoding 
+ [col for col in train_lr_data.columns if col.find('EDDisposition')!=-1 or col.find('CTDone')!=-1 or col.find('EDCT')!=-1 or col.find('PosCT')!=-1 ] 
)
downstream_missing_indicators = [f"{var}_missing" for var in downstream_var]

In [11]:
# first remove some clumns that we won't use in the model
remove_cols = identifier_cols+ ['AgeTwoPlus', 'AgeInMonth']\
            + list(set(downstream_var)) + downstream_missing_indicators
can_features = [col for col in train_lr_data.columns if col not in remove_cols]

In [12]:
len(can_features), len([col for col in can_features if col.endswith("_missing")])

(178, 35)

We have overall 180 candidate features with 35 missing indicators.

## Age < 2 

### Feature Selection

#### Univariate Screening

get rid of features 

+ almost constant (99\% same value)
+ risk difference is close to 0

In [13]:
# calculate how much the mode value occupies in the canidate features, to help me to remove nearly constant features
mode_rt_lt_2_df = {}
for col in can_features:
    mode_rt = train_lr_data_age_lt_2[col].value_counts(normalize=True, dropna=False).iloc[0]
    mode_rt_lt_2_df[col] = mode_rt
mode_rt_lt_2_df = pd.DataFrame.from_dict(mode_rt_lt_2_df, orient='index', columns=['mode_rt']).sort_values('mode_rt', ascending=False)
near_constant_threshold = 0.99 # if the mode proportion is over 99%, I assume it is nearly constant and carries little information
mode_rt_lt_2_df[mode_rt_lt_2_df['mode_rt'] > near_constant_threshold].head(10)

,mode_rt
Gender_missing,1.000000
HASeverity_3.0,1.000000
PosIntFinal_invalid,1.000000
AMSRepeat_1,1.000000
HAStart_4.0,1.000000
OSICut_1,0.999869
InjuryMech_3.0,0.999869
HAStart_3.0,0.999738
OSIPelvis_1,0.999738
HAStart_missing,0.999606


In [14]:
# calculate the risk difference for one single feature
risk_diff_lt_2_df = {}
for col in can_features:
    tmp = train_lr_data_age_lt_2[[target_col, col]]
    grp = tmp.groupby(col)[target_col].mean()
    if len(grp) >= 2:
        risk_diff = grp.max()-grp.min()
    risk_diff_lt_2_df[col] = risk_diff 
risk_diff_lt_2_df = pd.DataFrame.from_dict(risk_diff_lt_2_df, orient='index', columns=['risk_diff']).sort_values('risk_diff', ascending=True)
risk_nodiff_threshold = 0.01
risk_diff_lt_2_df[risk_diff_lt_2_df['risk_diff'] < risk_nodiff_threshold].head(10)

,risk_diff
VomitLast_3.0,0.000020
Gender,0.000207
Amnesia_verb_missing,0.001407
InjuryMech_8.0,0.002497
ClavFro_1,0.002562
ClavFace_1,0.002682
Drugs_missing,0.003028
Race_90.0,0.003222
VomitNbr_2.0,0.003826
Ethnicity_missing,0.005000


In [15]:
# combine the two logics together
def univariate_feature_screening(features,
                                 target_col,
                                 data,
                                 near_constant_threshold = 0.99,
                                 risk_nodiff_threshold = 0.01):
    omit_cols = set()
    
    for col in features:
        # A: near-constant
        mode_rt = data[col].value_counts(normalize=True, dropna=False).iloc[0]
        if mode_rt > near_constant_threshold:
            omit_cols.add(col)
            continue
        # B: little risk difference
        grp = data.groupby(col)[target_col].mean()
        if len(grp) >= 2:
            risk_diff = grp.max()-grp.min()
            if risk_diff < risk_nodiff_threshold:
                omit_cols.add(col)

    return omit_cols
    

In [16]:
omit_lt_2_cols = univariate_feature_screening(can_features, target_col, train_lr_data_age_lt_2)
omit_lt_2_cols

{'AMSRepeat_1',
 'AMSSlow_1',
 'AMS_missing',
 'AgeinYears',
 'Amnesia_verb_1.0',
 'Amnesia_verb_91.0',
 'Amnesia_verb_missing',
 'Clav',
 'ClavFace_1',
 'ClavFace_92',
 'ClavFro_1',
 'ClavFro_92',
 'ClavNeck_1',
 'ClavNeck_92',
 'ClavOcc_92',
 'ClavPar_92',
 'ClavTem_92',
 'Clav_missing',
 'Dizzy',
 'Drugs',
 'Drugs_missing',
 'Ethnicity',
 'Ethnicity_missing',
 'FontBulg',
 'Gender',
 'Gender_missing',
 'HASeverity_2.0',
 'HASeverity_3.0',
 'HASeverity_92.0',
 'HASeverity_missing',
 'HAStart_2.0',
 'HAStart_3.0',
 'HAStart_4.0',
 'HAStart_92.0',
 'HAStart_missing',
 'HA_verb_1.0',
 'HA_verb_91.0',
 'HA_verb_missing',
 'HemaLoc_missing',
 'HemaSize_2.0',
 'HemaSize_missing',
 'Hema_missing',
 'High_impact_InjSev_2.0',
 'InjuryMech_10.0',
 'InjuryMech_11.0',
 'InjuryMech_2.0',
 'InjuryMech_3.0',
 'InjuryMech_4.0',
 'InjuryMech_5.0',
 'InjuryMech_8.0',
 'InjuryMech_90.0',
 'Intubated',
 'Intubated_missing',
 'LocLen_4.0',
 'NeuroD',
 'NeuroDCranial_1',
 'NeuroDCranial_92',
 'NeuroDMotor

In [17]:
can_features_lt_2 = list(set(can_features).difference(omit_lt_2_cols))

In [18]:
len(can_features_lt_2)

66

#### Lasso Feature Selection

Use lasso (L1 Penalized) to do feature selection

In [19]:


feature_select_lt_2_lr = LogisticRegression(
    l1_ratio=1.0, # lasso regression
    solver="saga",
    class_weight="balanced", # important for imbalanced dataset
    C = 1.0,
    max_iter=5000,
    random_state=42
)
feature_select_lt_2_lr.fit(train_lr_data_age_lt_2[can_features_lt_2], train_lr_data_age_lt_2[target_col])


/Users/dannyzhou/anaconda3/envs/stat214/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",1.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:

In [20]:
selected_features_lt_2 = train_lr_data_age_lt_2[can_features_lt_2].columns[feature_select_lt_2_lr.coef_[0]!=0]

In [21]:
len(selected_features_lt_2)

66

It looks like, with lasso, we don't exclude new features.

In [22]:
selected_features_lt_2

Index(['HemaLoc_2.0', 'Vomit', 'Dizzy_missing', 'High_impact_InjSev_missing',
       'AMSOth_1', 'VomitStart_92.0', 'High_impact_InjSev_3.0', 'GCSMotor',
       'InjuryMech_12.0', 'SFxBas_missing', 'LocLen_3.0', 'LocLen_92.0',
       'InjuryMech_6.0', 'OSIExtremity_92', 'LOCSeparate_missing',
       'ActNorm_missing', 'GCSGroup', 'GCSTotal', 'AMSRepeat_92',
       'Race_missing', 'LOCSeparate_1.0', 'Seiz', 'InjuryMech_7.0',
       'ClavPar_1', 'OSIOth_92', 'NeuroD_missing', 'HemaSize_3.0',
       'SeizLen_92.0', 'HemaLoc_3.0', 'HemaSize_92.0', 'Hema', 'LocLen_2.0',
       'GCSVerbal', 'AMSOth_92', 'AMSSlow_92', 'HemaLoc_92.0', 'AMSSleep_92',
       'SeizOccur_92.0', 'OSIExtremity_1', 'ClavOcc_1', 'OSICspine_92',
       'SFxPalp_2.0', 'VomitLast_92.0', 'OSICut_92', 'LOCSeparate_2.0',
       'FontBulg_missing', 'AMS', 'OSIAbdomen_92', 'GCSEye', 'LocLen_missing',
       'VomitLast_missing', 'ActNorm', 'AMSAgitated_1', 'AMSAgitated_92',
       'Vomit_missing', 'AMSSleep_1', 'VomitNbr_3.0',

In [23]:
feature_names = train_lr_data_age_lt_2[can_features_lt_2].columns
coef = feature_select_lt_2_lr.coef_.ravel()   # shape (p,)

imp = (
    pd.DataFrame({"feature": feature_names, "coef": coef, "abs_coef": np.abs(coef)})
      .query("coef != 0")
      .sort_values("abs_coef", ascending=False)
)

imp.head(40)

,feature,coef,abs_coef
25,NeuroD_missing,18.876775,18.876775
7,GCSMotor,17.666313,17.666313
14,LOCSeparate_missing,16.906938,16.906938
28,HemaLoc_3.0,15.321873,15.321873
38,OSIExtremity_1,-13.971753,13.971753
11,LocLen_92.0,-13.567221,13.567221
4,AMSOth_1,-13.055840,13.055840
0,HemaLoc_2.0,12.363433,12.363433
21,Seiz,11.409301,11.409301
41,SFxPalp_2.0,11.218495,11.218495


Actually, out of the selected features, many of them are highly correlated or generated from one-hot encoding.

For example, a lot of features are ended with 92, which means their parent variable is set to `no`. In this case, I can use the simple binary parent variable.

I choose mannally to do some feature engineering, and settle down the final features for age<2 models.

+ GCS and its component: in this case, the exact component has predictive ability, I choose to keep the three component.
+ `IOS`
+ `HemaLoc_2.0`, `HemaLoc_3.0`
+ Mechnism: single mechnism like 3,6,9,12 is included, and high_impact_InjSev_3 (high) is included too. Based on the category of injury impact, I will choose keep `High_impact_InjSev_3.0`. Missingness also represents some information (High_impact_InjSev_missing and InjuryMech_missing ranks high). I choose to keep `High_impact_InjSev_missing`
+ `NeuroD_missing`
+ `AMS` and `AMSOth_1`: altered mental status especially those having other signs
+ `ActNorm` and `ActNorm_missing`
+ `ClavOcc_1` and `ClavPar_1`
+ `SFxPalp_2.0`
+ `Seiz`, 
+ `LOCSeparate_missing`
+ `VomitStart_3.0`

In [24]:
feature_lt_2_lr = ['GCSMotor', 'GCSVerbal','GCSEye','HemaLoc_2.0','HemaLoc_3.0','High_impact_InjSev_3.0',
                   'High_impact_InjSev_missing','NeuroD_missing','AMS','AMSOth_1','ActNorm','ActNorm_missing',
                   'ClavOcc_1','ClavPar_1','SFxPalp_2.0','Seiz','LOCSeparate_missing','VomitStart_3.0']

###  Modeling


#### Baseline

In [25]:
lt_2_lr = LogisticRegression(
    l1_ratio=1.0, # lasso regression
    solver="saga",
    class_weight="balanced",
    C=0.1,
    max_iter=20000,
    random_state=42
)

lt_2_lr.fit(train_lr_data_age_lt_2[feature_lt_2_lr], train_lr_data_age_lt_2[target_col])

/Users/dannyzhou/anaconda3/envs/stat214/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",0.1
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",1.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:

In [26]:

# define the metrics consistent with paper and some common metrics for imbalanced classification
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score
)

def pecarn_metrics(y_true, y_prob, threshold=0.02):
    """
    threshold = risk cutoff defining 'predictor present'
    """

    y_pred = (y_prob >= threshold).astype(int)

    TP = np.sum((y_true==1) & (y_pred==1))
    FN = np.sum((y_true==1) & (y_pred==0))
    TN = np.sum((y_true==0) & (y_pred==0))
    FP = np.sum((y_true==0) & (y_pred==1))

    sensitivity = TP / (TP + FN) if (TP+FN)>0 else np.nan
    npv = TN / (TN + FN) if (TN+FN)>0 else np.nan
    roc_auc = roc_auc_score(y_true, y_prob)
    pr_auc_score =average_precision_score(y_true, y_prob)
    precision = precision_score(y_true, y_pred)
    specificity = TN / (TN + FP) if (TN+FP)>0 else np.nan

    return {
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "NPV": npv,
        "ROC AUC": roc_auc,
        "PR AUC": pr_auc_score,
        "Precision":  precision,
        "TP": TP,
        "FN": FN,
        "TN": TN,
        "FP": FP
    }

In [27]:
# train 
train_lt_2_prob = lt_2_lr.predict_proba(train_lr_data_age_lt_2[feature_lt_2_lr])[:,1]
pecarn_metrics(train_lr_data_age_lt_2[target_col],train_lt_2_prob, threshold=0.5 )

{'Sensitivity': np.float64(0.9150943396226415),
 'Specificity': np.float64(0.8226214238190286),
 'NPV': np.float64(0.9985462768534971),
 'ROC AUC': 0.9301284224004821,
 'PR AUC': 0.3406751924521269,
 'Precision': 0.06783216783216783,
 'TP': np.int64(97),
 'FN': np.int64(9),
 'TN': np.int64(6182),
 'FP': np.int64(1333)}

In [28]:
# val 
val_lt_2_prob = lt_2_lr.predict_proba(val_lr_data_age_lt_2[feature_lt_2_lr])[:,1]
pecarn_metrics(val_lr_data_age_lt_2[target_col],val_lt_2_prob, threshold=0.5 )

{'Sensitivity': np.float64(0.9333333333333333),
 'Specificity': np.float64(0.839851024208566),
 'NPV': np.float64(0.9988925802879292),
 'ROC AUC': 0.945592799503414,
 'PR AUC': 0.31911757608390234,
 'Precision': 0.07526881720430108,
 'TP': np.int64(14),
 'FN': np.int64(1),
 'TN': np.int64(902),
 'FP': np.int64(172)}

In [29]:
# find the best threshold 
from sklearn.metrics import confusion_matrix
def scan_thresholds(y_true, y_prob,
                    thresholds=np.linspace(0.001, 0.2, 200)):
    rows = []
    y_true = np.asarray(y_true).astype(int)

    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

        sens = tp / (tp + fn) if (tp + fn) else np.nan
        spec = tn / (tn + fp) if (tn + fp) else np.nan
        rows.append({"thr": t, "sensitivity": sens, "specificity": spec,
                     "youdenJ": sens + spec - 1})
    return pd.DataFrame(rows)


In [30]:
thr_df = scan_thresholds(
    val_lr_data_age_lt_2[target_col],
    val_lt_2_prob
)
thr_df.sort_values("youdenJ", ascending=False).head(10)

,thr,sensitivity,specificity,youdenJ
199,0.200,0.933333,0.799814,0.733147
198,0.199,0.933333,0.799814,0.733147
197,0.198,0.933333,0.799814,0.733147
193,0.194,0.933333,0.795158,0.728492
191,0.192,0.933333,0.795158,0.728492
192,0.193,0.933333,0.795158,0.728492
190,0.191,0.933333,0.795158,0.728492
194,0.195,0.933333,0.795158,0.728492
196,0.197,0.933333,0.795158,0.728492
195,0.196,0.933333,0.795158,0.728492


In [31]:
# to find the best threshold: sensitivity > 0.95 and then among those thresholds, find the one with the highest specificity
thr_df[(thr_df['sensitivity']>=0.95)].sort_values("specificity", ascending=False).head(10)

,thr,sensitivity,specificity,youdenJ
25,0.026,1.0,0.703911,0.703911
24,0.025,1.0,0.703911,0.703911
23,0.024,1.0,0.703911,0.703911
22,0.023,1.0,0.703911,0.703911
21,0.022,1.0,0.703911,0.703911
20,0.021,1.0,0.703911,0.703911
19,0.020,1.0,0.703911,0.703911
18,0.019,1.0,0.703911,0.703911
17,0.018,1.0,0.703911,0.703911
16,0.017,1.0,0.702048,0.702048


**Remark:**

For the baseline model, as for NPV, it can achieve 99\% at almost every threshold.

It looks like, our baseline model has achieved enough negative predictive value, although it was worse in sensitivity compared with CDR

#### Hyperparameter Tuning

Two hyperparameters:

+ C (regularization strength)
+ class_weight


In [32]:
# implement grid search manually, as for the criterion, I will use sensitivity with threshold 0.5 
Cs = [0.01, 0.05, 0.1, 0.5, 1]
weights = [5,10,20,50]

lr_lt_2_results = []

for C in Cs:
    for w in weights:

        lr = LogisticRegression(
            l1_ratio = 1,
            solver="saga",
            class_weight={0:1, 1:w},
            C=C,
            max_iter=20000,
            random_state=42
        )

        lr.fit(train_lr_data_age_lt_2[feature_lt_2_lr], train_lr_data_age_lt_2[target_col])
        prob = lr.predict_proba(val_lr_data_age_lt_2[feature_lt_2_lr])[:,1]

        lr_lt_2_results.append({
            "C": C,
            "weight": w,
            "PR_AUC": average_precision_score(val_lr_data_age_lt_2[target_col], prob),
            'Sensitivity': recall_score(val_lr_data_age_lt_2[target_col], prob>0.5)
        })

/Users/dannyzhou/anaconda3/envs/stat214/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/dannyzhou/anaconda3/envs/stat214/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/dannyzhou/anaconda3/envs/stat214/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/dannyzhou/anaconda3/envs/stat214/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/dannyzhou/anaconda3/envs/stat214/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not conver

In [33]:
pd.DataFrame(lr_lt_2_results).sort_values( by=['Sensitivity', 'PR_AUC'], ascending= [ False, False])

,C,weight,PR_AUC,Sensitivity
11,0.10,50,0.283121,0.933333
7,0.05,50,0.221951,0.933333
19,1.00,50,0.211414,0.933333
3,0.01,50,0.351499,0.866667
10,0.10,20,0.328657,0.733333
14,0.50,20,0.282799,0.733333
15,0.50,50,0.237020,0.733333
18,1.00,20,0.333753,0.600000
6,0.05,20,0.297615,0.533333
9,0.10,10,0.316220,0.466667


The optimal combination:

+ C: 0.1
+ class weight: 50

#### Best model

In [34]:
best_lt_2_lr= LogisticRegression(
    l1_ratio=1,
    solver="saga",
    class_weight={0:1, 1:50},
    C=0.1,
    max_iter=20000,
    random_state=42
)

best_lt_2_lr.fit(train_lr_data_age_lt_2[feature_lt_2_lr], train_lr_data_age_lt_2[target_col])

/Users/dannyzhou/anaconda3/envs/stat214/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",0.1
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",1
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*","{0: 1, 1: 50}"
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :te

In [35]:
# train 
best_train_lt_2_prob = best_lt_2_lr.predict_proba(train_lr_data_age_lt_2[feature_lt_2_lr])[:,1]
pecarn_metrics(train_lr_data_age_lt_2[target_col],best_train_lt_2_prob, threshold=0.5 )

{'Sensitivity': np.float64(0.9150943396226415),
 'Specificity': np.float64(0.8054557551563539),
 'NPV': np.float64(0.9985153414714616),
 'ROC AUC': 0.9321595802106478,
 'PR AUC': 0.3685151022446466,
 'Precision': 0.062219371391917896,
 'TP': np.int64(97),
 'FN': np.int64(9),
 'TN': np.int64(6053),
 'FP': np.int64(1462)}

In [36]:
# val 
best_val_lt_2_prob = best_lt_2_lr.predict_proba(val_lr_data_age_lt_2[feature_lt_2_lr])[:,1]
pecarn_metrics(val_lr_data_age_lt_2[target_col],best_val_lt_2_prob, threshold=0.5 )

{'Sensitivity': np.float64(0.9333333333333333),
 'Specificity': np.float64(0.8184357541899442),
 'NPV': np.float64(0.9988636363636364),
 'ROC AUC': 0.9364680322780881,
 'PR AUC': 0.2831212637025784,
 'Precision': 0.06698564593301436,
 'TP': np.int64(14),
 'FN': np.int64(1),
 'TN': np.int64(879),
 'FP': np.int64(195)}

In [37]:
# use validation set to choose the optimal threshold
thr_best_df = scan_thresholds(
    val_lr_data_age_lt_2[target_col],
    best_val_lt_2_prob
)
thr_best_df[(thr_best_df['sensitivity']>=0.95)].sort_values("specificity", ascending=False).head(10)

,thr,sensitivity,specificity,youdenJ
20,0.021,1.0,0.723464,0.723464
19,0.020,1.0,0.722533,0.722533
18,0.019,1.0,0.722533,0.722533
17,0.018,1.0,0.722533,0.722533
16,0.017,1.0,0.722533,0.722533
15,0.016,1.0,0.722533,0.722533
14,0.015,1.0,0.720670,0.720670
13,0.014,1.0,0.720670,0.720670
12,0.013,1.0,0.692737,0.692737
11,0.012,1.0,0.689944,0.689944


With threshold at around **0.02**, we can achieve 100\% sensitivity with the highest specifity.

In [38]:
# train set
best_train_lt_2_prob = best_lt_2_lr.predict_proba(train_lr_data_age_lt_2[feature_lt_2_lr])[:,1]
pecarn_metrics(train_lr_data_age_lt_2[target_col],best_train_lt_2_prob, threshold=0.02 )

{'Sensitivity': np.float64(0.9339622641509434),
 'Specificity': np.float64(0.7088489687292082),
 'NPV': np.float64(0.9986876640419947),
 'ROC AUC': 0.9321595802106478,
 'PR AUC': 0.3685151022446466,
 'Precision': 0.043288150415391344,
 'TP': np.int64(99),
 'FN': np.int64(7),
 'TN': np.int64(5327),
 'FP': np.int64(2188)}

In [39]:
# val set
best_val_lt_2_prob = best_lt_2_lr.predict_proba(val_lr_data_age_lt_2[feature_lt_2_lr])[:,1]
pecarn_metrics(val_lr_data_age_lt_2[target_col],best_val_lt_2_prob, threshold=0.02 )

{'Sensitivity': np.float64(1.0),
 'Specificity': np.float64(0.7225325884543762),
 'NPV': np.float64(1.0),
 'ROC AUC': 0.9364680322780881,
 'PR AUC': 0.2831212637025784,
 'Precision': 0.04792332268370607,
 'TP': np.int64(15),
 'FN': np.int64(0),
 'TN': np.int64(776),
 'FP': np.int64(298)}

In [40]:
# evalutate on test set
best_test_lt_2_prob = best_lt_2_lr.predict_proba(test_lr_data_age_lt_2[feature_lt_2_lr])[:,1]
pecarn_metrics(test_lr_data_age_lt_2[target_col],best_test_lt_2_prob, threshold=0.02 )

{'Sensitivity': np.float64(0.8),
 'Specificity': np.float64(0.7079646017699115),
 'NPV': np.float64(0.9960681520314548),
 'ROC AUC': 0.8416239714330073,
 'PR AUC': 0.26119335425423534,
 'Precision': 0.03686635944700461,
 'TP': np.int64(24),
 'FN': np.int64(6),
 'TN': np.int64(1520),
 'FP': np.int64(627)}

### Interpretability

With the threshold at 0.02, we can achieve 0.8 sensitivity and 0.9961 negative predictive value on our test set.

Let's inspect the selected features and look into how they can be related to the real clinical world

In [41]:
coef = pd.Series(best_lt_2_lr.coef_[0], index=feature_lt_2_lr)
coef.sort_values(key=np.abs, ascending=False).head(15)

Seiz                   18.713015
NeuroD_missing         14.625928
GCSEye                 -9.219562
ClavOcc_1               9.018224
GCSVerbal              -8.105043
LOCSeparate_missing     8.088586
HemaLoc_3.0             7.597902
GCSMotor                6.256959
SFxPalp_2.0             5.852919
ActNorm                -5.848910
AMS                     5.527215
HemaLoc_2.0             5.304370
ClavPar_1               5.196593
VomitStart_3.0          5.126908
ActNorm_missing         4.742773
dtype: float64

**Remark:** 

we examined the magnitude and direction of the logistic regression coefficients selected after feature selection. Because logistic regression models the log-odds of clinically important traumatic brain injury (ciTBI), positive coefficients indicate increased risk while negative coefficients indicate protective or low-risk signals.

Different from the CDR model, we include the GCS componet `GCSEye`, `GCSVerbal` and `GCSMotor` in our model. The first two have large and negative coefficients in our model, indicating that higher neurologic function strongly reduces predicted risk, consistent with established clinical knowledge. While for `GCSMotor`, it has positive coefficient, which is contrary to our clinical knowlegde. It looks like things become more complicated when it comes to motor part.

Conversely, altered mental status, seizure, severe injury mechanism, and signs suggestive of skull fracture are associated with increased risk, reflecting patterns emphasized in the PECARN decision rule.

Interestingly, several missingness indicators also appear among the influential features. Rather than behaving as random noise, missing clinical assessments are associated with elevated predicted risk. This likely reflects real-world emergency department workflow, where incomplete documentation often occurs in more severe or unstable patients, suggesting that missingness itself carries clinical information.


## Age >= 2

### Feature Selection

#### Univariate Screening

In [42]:
omit_gte_2_cols = univariate_feature_screening(can_features, target_col, train_lr_data_age_gte_2)
omit_gte_2_cols

{'AMS_missing',
 'ClavFace_1',
 'ClavFro_1',
 'ClavOcc_1',
 'Clav_missing',
 'Dizzy',
 'Drugs_missing',
 'Ethnicity',
 'Ethnicity_missing',
 'FontBulg',
 'FontBulg_missing',
 'Gender_missing',
 'HASeverity_2.0',
 'HASeverity_92.0',
 'HAStart_2.0',
 'HAStart_4.0',
 'HAStart_92.0',
 'HA_verb_1.0',
 'HemaLoc_2.0',
 'HemaLoc_missing',
 'HemaSize_2.0',
 'Hema_missing',
 'High_impact_InjSev_missing',
 'InjuryMech_4.0',
 'InjuryMech_8.0',
 'InjuryMech_90.0',
 'InjuryMech_missing',
 'Intubated',
 'Intubated_missing',
 'LocLen_2.0',
 'LocLen_4.0',
 'NeuroDCranial_1',
 'NeuroDMotor_1',
 'NeuroDOth_1',
 'NeuroDReflex_1',
 'NeuroDSensory_1',
 'OSICspine_1',
 'OSICut_1',
 'OSIPelvis_1',
 'OSI_missing',
 'Paralyzed',
 'Paralyzed_missing',
 'PosIntFinal_invalid',
 'Race_3.0',
 'Race_4.0',
 'Race_5.0',
 'Race_90.0',
 'Race_missing',
 'SFxBasHem_1',
 'SFxBasOto_1',
 'SFxBasPer_1',
 'SFxBasRet_1',
 'SFxBasRhi_1',
 'SFxPalpDepress_1.0',
 'SFxPalpDepress_92.0',
 'SFxPalpDepress_missing',
 'SFxPalp_1.0',
 

In [43]:
len(omit_gte_2_cols)

74

In [44]:
can_features_gte_2 = list(set(can_features).difference(omit_gte_2_cols))
len(can_features_gte_2)

104

#### Lasso Feature Selection

In [45]:
feature_select_gte_2_lr = LogisticRegression(
    l1_ratio = 1,
    solver="saga",
    class_weight="balanced", # important for imbalanced dataset
    C = 1.0,
    max_iter=5000,
    random_state=42
)
feature_select_gte_2_lr.fit(train_lr_data_age_gte_2[can_features_gte_2], train_lr_data_age_gte_2[target_col])

/Users/dannyzhou/anaconda3/envs/stat214/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",1
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`m

In [46]:
feature_names = train_lr_data_age_gte_2[can_features_gte_2].columns
coef = feature_select_gte_2_lr.coef_.ravel()   # shape (p,)

imp = (
    pd.DataFrame({"feature": feature_names, "coef": coef, "abs_coef": np.abs(coef)})
      .query("coef != 0")
      .sort_values("abs_coef", ascending=False)
)

imp[imp['abs_coef']>=1]

,feature,coef,abs_coef
54,SFxBas,12.692387,12.692387
24,High_impact_InjSev_3.0,7.094777,7.094777
45,InjuryMech_11.0,-5.902524,5.902524
14,NeuroD,4.909552,4.909552
58,GCSGroup,-4.647114,4.647114
96,LocLen_missing,3.984051,3.984051
44,AMS,3.877752,3.877752
103,InjuryMech_10.0,-3.708406,3.708406
13,SFxPalp_2.0,3.680026,3.680026
28,HAStart_3.0,-3.431815,3.431815


Still, I manually do the feature selection to remove redundancy:

+ `SFxBas`
+ `GCSGroup`
+ `High_impact_InjSev_2.0`, `High_impact_InjSev_3.0`: There are exact mechnims like 11, 3,10,5 and etc. showing large coefficient, but they can be mostly inclued in injury impact at level 2 or 3.
+ `NeuroD`, `NeuroD_missing	`
+ `OSIOth_1.0`
+ `AMSAgitated_1.0`
+ `SFxPalp_2.0`
+ `AMS`
+ `ClavPar_1.0`
+ `LocLen_missing` and `LOCSeparate_missing`
+ `ClavNeck_1.0`
+ `HAStart_3.0`, `HAStart_missing`
+ `Vomit`, `VomitStart_2.0`, `VomitLast_missing`, `VomitStart_3.0`
+ `ActNorm`, `ActNorm_missing`
+ `Dizzy_missing`
+ `HemaSize_3.0`
+ `HASeverity_3.0`

In [47]:
feature_gte_2_lr =  [
    "SFxBas",
    "GCSGroup",

    "High_impact_InjSev_2.0",
    "High_impact_InjSev_3.0",

    "NeuroD",
    "NeuroD_missing",

    "OSIOth_1",
    "AMSAgitated_1",
    "SFxPalp_2.0",
    "AMS",

    "ClavPar_1",

    "LocLen_missing",
    "LOCSeparate_missing",

    "ClavNeck_1",

    "HAStart_3.0",
    "HAStart_missing",

    "Vomit",
    "VomitStart_2.0",
    "VomitLast_missing",
    "VomitStart_3.0",

    "ActNorm",
    "ActNorm_missing",

    "Dizzy_missing",

    "HemaSize_3.0",
    "HASeverity_3.0"
]

### Modeling

#### Baseline

In [48]:
gte_2_lr = LogisticRegression(
    l1_ratio=1,
    solver="saga",
    class_weight="balanced",
    C=0.1,
    max_iter=20000,
    random_state=42
)

gte_2_lr.fit(train_lr_data_age_gte_2[feature_gte_2_lr], train_lr_data_age_gte_2[target_col])

/Users/dannyzhou/anaconda3/envs/stat214/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",0.1
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",1
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`m

In [49]:
# train 
train_gte_2_prob = gte_2_lr.predict_proba(train_lr_data_age_gte_2[feature_gte_2_lr])[:,1]
pecarn_metrics(train_lr_data_age_gte_2[target_col],train_gte_2_prob, threshold=0.5 )

{'Sensitivity': np.float64(0.9176755447941889),
 'Specificity': np.float64(0.7396571838822579),
 'NPV': np.float64(0.997941639423659),
 'ROC AUC': 0.9236800853704831,
 'PR AUC': 0.34770084489404446,
 'Precision': 0.06131693900663323,
 'TP': np.int64(379),
 'FN': np.int64(34),
 'TN': np.int64(16484),
 'FP': np.int64(5802)}

In [50]:
# val
val_gte_2_prob = gte_2_lr.predict_proba(val_lr_data_age_gte_2[feature_gte_2_lr])[:,1]
pecarn_metrics(val_lr_data_age_gte_2[target_col],val_gte_2_prob, threshold=0.5 )

{'Sensitivity': np.float64(0.896551724137931),
 'Specificity': np.float64(0.7399497487437185),
 'NPV': np.float64(0.9974597798475868),
 'ROC AUC': 0.9234751342921503,
 'PR AUC': 0.4184054360657538,
 'Precision': 0.05909090909090909,
 'TP': np.int64(52),
 'FN': np.int64(6),
 'TN': np.int64(2356),
 'FP': np.int64(828)}

### Hyperparamter Tuning

In this part, I will use random search instead of grid search due to the time-complexity from  larger sample sizes and more features.

In [51]:
rng = np.random.default_rng(42)

N_TRIALS = 20   # instead of len(Cs)*len(weights)

results = []

for _ in range(N_TRIALS):


    C = 10 ** rng.uniform(-2, 0)   # log-uniform: 0.01 → 1
    w = rng.choice([5,10,20,50,100])

    lr = LogisticRegression(
        l1_ratio=1,
        solver="saga",
        class_weight={0:1, 1:int(w)},
        C=C,
        max_iter=8000,     # reduced safely
        random_state=42
    )

    lr.fit(
        train_lr_data_age_gte_2[feature_gte_2_lr],
        train_lr_data_age_gte_2[target_col]
    )

    prob = lr.predict_proba(
        val_lr_data_age_gte_2[feature_gte_2_lr]
    )[:, 1]

    sens = recall_score(
        val_lr_data_age_gte_2[target_col],
        prob >= 0.5
    )

    results.append({
        "C": C,
        "weight": w,
        "Sensitivity": sens,
        "PR_AUC": average_precision_score(
            val_lr_data_age_gte_2[target_col], prob
        )
    })

lr_gte_2_results = pd.DataFrame(results).sort_values(
    "Sensitivity", ascending=False
)

/Users/dannyzhou/anaconda3/envs/stat214/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/dannyzhou/anaconda3/envs/stat214/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/dannyzhou/anaconda3/envs/stat214/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/dannyzhou/anaconda3/envs/stat214/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/dannyzhou/anaconda3/envs/stat214/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not conver

In [52]:
lr_gte_2_results 

,C,weight,Sensitivity,PR_AUC
9,0.077060,100,0.965517,0.185968
13,0.328228,50,0.965517,0.221798
15,0.611283,100,0.931034,0.155484
4,0.332874,50,0.931034,0.401782
0,0.353112,50,0.913793,0.332249
5,0.018040,50,0.896552,0.423238
16,0.360385,50,0.896552,0.368190
11,0.013416,20,0.862069,0.507236
6,0.079574,20,0.810345,0.516617
8,0.193968,20,0.810345,0.494357


Combining the sensitivity and PR_AUC, C=0.3 and weight = 50 is the best choice.

As for the weight, I think it's very reasonable, since we have very low positive ratio, adding more weights on them is necessary.

### Best Model

In [53]:
best_gte_2_lr= LogisticRegression(
    l1_ratio=1,
    solver="saga",
    class_weight={0:1, 1:50},
    C=0.3,
    max_iter=20000,
    random_state=42
)

best_gte_2_lr.fit(train_lr_data_age_gte_2[feature_gte_2_lr], train_lr_data_age_gte_2[target_col])

/Users/dannyzhou/anaconda3/envs/stat214/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",0.3
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",1
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*","{0: 1, 1: 50}"
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :te

In [54]:
# train 
best_train_gte_2_prob = best_gte_2_lr.predict_proba(train_lr_data_age_gte_2[feature_gte_2_lr])[:,1]
pecarn_metrics(train_lr_data_age_gte_2[target_col],best_train_gte_2_prob, threshold=0.5 )

{'Sensitivity': np.float64(0.6924939467312349),
 'Specificity': np.float64(0.9363277393879565),
 'NPV': np.float64(0.9939506525674002),
 'ROC AUC': 0.9361907898182096,
 'PR AUC': 0.3835353743221267,
 'Precision': 0.16774193548387098,
 'TP': np.int64(286),
 'FN': np.int64(127),
 'TN': np.int64(20867),
 'FP': np.int64(1419)}

In [55]:
# val
best_val_gte_2_prob = best_gte_2_lr.predict_proba(val_lr_data_age_gte_2[feature_gte_2_lr])[:,1]
pecarn_metrics(val_lr_data_age_gte_2[target_col],best_val_gte_2_prob, threshold=0.5 )

{'Sensitivity': np.float64(0.7931034482758621),
 'Specificity': np.float64(0.9340452261306532),
 'NPV': np.float64(0.9959812458137978),
 'ROC AUC': 0.950003249003639,
 'PR AUC': 0.407577440350045,
 'Precision': 0.1796875,
 'TP': np.int64(46),
 'FN': np.int64(12),
 'TN': np.int64(2974),
 'FP': np.int64(210)}

find the best threshold

In [56]:
# use validation set to choose the optimal threshold
thr_best_df = scan_thresholds(
    val_lr_data_age_gte_2[target_col],
    best_val_gte_2_prob
)

In [57]:
thr_best_df.sort_values(by= ['sensitivity','specificity'], ascending=[False, False]).head(10)

,thr,sensitivity,specificity,youdenJ
117,0.118,0.810345,0.928706,0.739051
118,0.119,0.810345,0.928706,0.739051
98,0.099,0.810345,0.928078,0.738423
99,0.100,0.810345,0.928078,0.738423
100,0.101,0.810345,0.928078,0.738423
101,0.102,0.810345,0.928078,0.738423
102,0.103,0.810345,0.928078,0.738423
103,0.104,0.810345,0.928078,0.738423
104,0.105,0.810345,0.928078,0.738423
105,0.106,0.810345,0.928078,0.738423


Actually, our model performance changes a little as threshold changes. Anyways, choose **0.1** as the optimal threshold.

In [58]:
# test
best_test_gte_2_prob = best_gte_2_lr.predict_proba(test_lr_data_age_gte_2[feature_gte_2_lr])[:,1]
pecarn_metrics(test_lr_data_age_gte_2[target_col],best_test_gte_2_prob, threshold=0.1)

{'Sensitivity': np.float64(0.7692307692307693),
 'Specificity': np.float64(0.9331240188383045),
 'NPV': np.float64(0.995478144364428),
 'ROC AUC': 0.9416589515490614,
 'PR AUC': 0.3788761642633354,
 'Precision': 0.1744186046511628,
 'TP': np.int64(90),
 'FN': np.int64(27),
 'TN': np.int64(5944),
 'FP': np.int64(426)}

In [59]:
# train
best_train_gte_2_prob = best_gte_2_lr.predict_proba(train_lr_data_age_gte_2[feature_gte_2_lr])[:,1]
pecarn_metrics(train_lr_data_age_gte_2[target_col],best_train_gte_2_prob, threshold=0.1)

{'Sensitivity': np.float64(0.7288135593220338),
 'Specificity': np.float64(0.9268150408328099),
 'NPV': np.float64(0.9946068281408003),
 'ROC AUC': 0.9361907898182096,
 'PR AUC': 0.3835353743221267,
 'Precision': 0.15579710144927536,
 'TP': np.int64(301),
 'FN': np.int64(112),
 'TN': np.int64(20655),
 'FP': np.int64(1631)}

In [60]:
# val
best_val_gte_2_prob = best_gte_2_lr.predict_proba(val_lr_data_age_gte_2[feature_gte_2_lr])[:,1]
pecarn_metrics(val_lr_data_age_gte_2[target_col],best_val_gte_2_prob, threshold=0.1)

{'Sensitivity': np.float64(0.8103448275862069),
 'Specificity': np.float64(0.9280778894472361),
 'NPV': np.float64(0.9962913014160486),
 'ROC AUC': 0.950003249003639,
 'PR AUC': 0.407577440350045,
 'Precision': 0.17028985507246377,
 'TP': np.int64(47),
 'FN': np.int64(11),
 'TN': np.int64(2955),
 'FP': np.int64(229)}

**Remark:**

Our final best model only acheives sensitivity of 0.7692 and NPV of 0.9954, which is a little worse than CDR model for group aged at 2 and older.

### Interpretability

In [61]:
coef = pd.Series(best_gte_2_lr.coef_[0], index=feature_gte_2_lr)
coef.sort_values(key=np.abs, ascending=False).head(20)

GCSGroup                 -18.259415
SFxBas                    16.214518
NeuroD                    13.955233
AMS                       11.908855
LocLen_missing            11.110353
VomitStart_3.0            10.424298
ActNorm                  -10.420273
HAStart_3.0               -8.653041
ClavPar_1                  7.724138
OSIOth_1                   7.686336
VomitStart_2.0             7.221127
VomitLast_missing         -6.843690
AMSAgitated_1             -6.710342
NeuroD_missing            -6.078166
HemaSize_3.0               6.054260
Vomit                      5.915072
LOCSeparate_missing        5.638590
High_impact_InjSev_3.0     5.129602
Dizzy_missing             -4.817694
ActNorm_missing            3.238351
dtype: float64

For children aged ≥2 years, the logistic regression model identifies clinically meaningful predictors that align closely with established traumatic brain injury risk factors. 

The strongest positive associations with ciTBI risk include signs of basilar skull fracture (SFxBas), neurological deficit (NeuroD), altered mental status (AMS), severe headache (HASeverity), and high-impact injury mechanisms, all of which are consistent with known high-risk clinical indicators. 

Lower Glasgow Coma Scale status (GCSGroup) shows a strong negative coefficient, indicating that better neurological status substantially reduces predicted risk. It is consistent with CDR modeling, GCSGroup =1 for GCS 3-13, GCSGroup =2 for GCS 14-15.

Several missingness indicators (e.g., Dizzy_missing, LOCSeparate_missing, LocLen_missing) also carry positive coefficients, suggesting that incomplete assessments themselves may reflect higher clinical concern or workflow-driven selective documentation. 

## Stability

One judgment call in the logistic regression model was imputing missing Glasgow Coma Scale (GCS) components using clinically informed rules. Because GCS variables are central predictors, we evaluated whether model performance depended on this assumption.

To test stability, we repeated the entire modeling procedure under a stricter scenario where all observations with missing GCS components were removed instead of imputed. The same feature set, training procedure, and evaluation metrics were used.

Model performance was then compared between:

+ Imputed dataset (primary analysis)
+ Complete-case dataset (no GCS imputation)

Consistent performance across the two settings indicates that the selected predictors and model conclusions are robust to the GCS imputation strategy.


In [62]:
# prepare another version of the data for stability check 

train_lr_data_new, val_lr_data_new, test_lr_data_new = preprocess_data(
    df=tbi_cleaned_new,
    numeric_cols=numeric_cols,
    categorical_cols=categorical_cols,
    multi_category_cols=multi_category_cols,
    binary_cols=binary_cols,
    parent_child_groups=parent_child_groups,
    target_col='PosIntFinal',
    test_size=0.2,
    val_size=0.1,
    random_state=42,
    stratify_col=['PosIntFinal', 'AgeTwoPlus'],
    if_exclude_gcs_under_13= False, # include GCS 3-13 different from paper
    if_fill_missing_gcs=False,
    gcs_fill_strategy="leave", # leave missing 
    if_one_hot_encode=True,
    drop_first_cat_in_ohe=True,
    if_drop_na_rows=True, # after all the missing values are handled, drop any remaining rows with NA
    if_handle_parent_child_missing=True,
    if_handle_missing_for_categories=True,
    missing_category_label="missing",
    if_add_flag_missing_cols=True, # add flag columns to indicate missingness for each feature
    parent_missing_strategy="fill0",
    child_missing_when_parent_yes="missing_category",
    binary_missing_strategy="fill0",
    multi_missing_strategy="missing_category"
)

In [63]:
train_lr_data_new.shape, val_lr_data_new.shape, test_lr_data_new.shape

((29408, 290), (4193, 290), (8437, 290))

In [64]:
# compare the rows in the two versions of the data
tot_rows = train_lr_data.shape[0] + val_lr_data.shape[0] + test_lr_data.shape[0]
new_tot_rows = train_lr_data_new.shape[0] + val_lr_data_new.shape[0] + test_lr_data_new.shape[0]
print(f"Original total rows: {tot_rows}, New total rows: {new_tot_rows}")

Original total rows: 43315, New total rows: 42038


In [65]:
# split the data into age < 2 and age >= 2 
train_lr_data_age_lt_2_new = train_lr_data_new[train_lr_data_new['AgeTwoPlus'] == 1]
train_lr_data_age_gte_2_new = train_lr_data_new[train_lr_data_new['AgeTwoPlus'] == 2]
val_lr_data_age_lt_2_new = val_lr_data_new[val_lr_data_new['AgeTwoPlus'] == 1]
val_lr_data_age_gte_2_new = val_lr_data_new[val_lr_data_new['AgeTwoPlus'] == 2]
test_lr_data_age_lt_2_new = test_lr_data_new[test_lr_data_new['AgeTwoPlus'] == 1]
test_lr_data_age_gte_2_new = test_lr_data_new[test_lr_data_new['AgeTwoPlus'] == 2]

We drop 1277 rows in the stricted dataset in all.

### Age < 2

use the same features, hyperparamters and thresholds on the stricted datasets

In [66]:
best_lt_2_lr_new= LogisticRegression(
    l1_ratio=1,
    solver="saga",
    class_weight={0:1, 1:50},
    C=0.1,
    max_iter=20000,
    random_state=42
)
best_lt_2_lr_new.fit(train_lr_data_age_lt_2_new[feature_lt_2_lr], train_lr_data_age_lt_2_new[target_col])

/Users/dannyzhou/anaconda3/envs/stat214/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",0.1
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",1
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*","{0: 1, 1: 50}"
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :te

In [67]:
# train 
best_train_lt_2_prob_new = best_lt_2_lr_new.predict_proba(train_lr_data_age_lt_2_new[feature_lt_2_lr])[:,1]
pecarn_metrics(train_lr_data_age_lt_2_new[target_col],best_train_lt_2_prob_new, threshold=0.02 )

{'Sensitivity': np.float64(0.9223300970873787),
 'Specificity': np.float64(0.7404748757592491),
 'NPV': np.float64(0.9985107967237528),
 'ROC AUC': 0.9287934306530212,
 'PR AUC': 0.297830372500582,
 'Precision': 0.04810126582278481,
 'TP': np.int64(95),
 'FN': np.int64(8),
 'TN': np.int64(5364),
 'FP': np.int64(1880)}

In [68]:
# val 
best_val_lt_2_prob_new = best_lt_2_lr_new.predict_proba(val_lr_data_age_lt_2_new[feature_lt_2_lr])[:,1]
pecarn_metrics(val_lr_data_age_lt_2_new[target_col],best_val_lt_2_prob_new, threshold=0.02 )

{'Sensitivity': np.float64(1.0),
 'Specificity': np.float64(0.7616731517509727),
 'NPV': np.float64(1.0),
 'ROC AUC': 0.9281652199940138,
 'PR AUC': 0.14225111802007553,
 'Precision': 0.050387596899224806,
 'TP': np.int64(13),
 'FN': np.int64(0),
 'TN': np.int64(783),
 'FP': np.int64(245)}

In [69]:
# test
best_val_lt_2_prob_new = best_lt_2_lr_new.predict_proba(val_lr_data_age_lt_2_new[feature_lt_2_lr])[:,1]
pecarn_metrics(val_lr_data_age_lt_2_new[target_col],best_val_lt_2_prob_new, threshold=0.02 )

{'Sensitivity': np.float64(1.0),
 'Specificity': np.float64(0.7616731517509727),
 'NPV': np.float64(1.0),
 'ROC AUC': 0.9281652199940138,
 'PR AUC': 0.14225111802007553,
 'Precision': 0.050387596899224806,
 'TP': np.int64(13),
 'FN': np.int64(0),
 'TN': np.int64(783),
 'FP': np.int64(245)}

**Remark:**

The selected feature and logistic regression is stable respecting to the GSC imputation. Re-training the model on a stricter dataset still shows excellent performance with 100\% sensistivity and 100\% NPV.

### Age >=2

In [70]:
best_gte_2_lr_new= LogisticRegression(
    l1_ratio=1,
    solver="saga",
    class_weight={0:1, 1:50},
    C=0.3,
    max_iter=20000,
    random_state=42
)

best_gte_2_lr_new.fit(train_lr_data_age_gte_2_new[feature_gte_2_lr], train_lr_data_age_gte_2_new[target_col])

/Users/dannyzhou/anaconda3/envs/stat214/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",0.3
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",1
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*","{0: 1, 1: 50}"
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :te

In [71]:
# train
best_train_gte_2_prob_new = best_gte_2_lr_new.predict_proba(train_lr_data_age_gte_2_new[feature_gte_2_lr])[:,1]
pecarn_metrics(train_lr_data_age_gte_2_new[target_col],best_train_gte_2_prob_new, threshold=0.1 )

{'Sensitivity': np.float64(0.9002493765586035),
 'Specificity': np.float64(0.7579409048938135),
 'NPV': np.float64(0.9975694233456888),
 'ROC AUC': 0.9194795789842106,
 'PR AUC': 0.24740289522887185,
 'Precision': 0.06441827266238401,
 'TP': np.int64(361),
 'FN': np.int64(40),
 'TN': np.int64(16417),
 'FP': np.int64(5243)}

In [72]:
# val
best_val_gte_2_prob_new = best_gte_2_lr_new.predict_proba(val_lr_data_age_gte_2_new[feature_gte_2_lr])[:,1]
pecarn_metrics(val_lr_data_age_gte_2_new[target_col],best_val_gte_2_prob_new, threshold=0.1 )

{'Sensitivity': np.float64(0.875),
 'Specificity': np.float64(0.7674418604651163),
 'NPV': np.float64(0.9970625262274444),
 'ROC AUC': 0.9093703857511998,
 'PR AUC': 0.230675069785024,
 'Precision': 0.06371911573472042,
 'TP': np.int64(49),
 'FN': np.int64(7),
 'TN': np.int64(2376),
 'FP': np.int64(720)}

In [73]:
# test
best_test_gte_2_prob_new = best_gte_2_lr_new.predict_proba(test_lr_data_age_gte_2_new[feature_gte_2_lr])[:,1]
pecarn_metrics(test_lr_data_age_gte_2_new[target_col],best_test_gte_2_prob_new, threshold=0.1 )

{'Sensitivity': np.float64(0.9122807017543859),
 'Specificity': np.float64(0.7662588538312942),
 'NPV': np.float64(0.9979035639412998),
 'ROC AUC': 0.9260867477773636,
 'PR AUC': 0.2990895959968859,
 'Precision': 0.06683804627249357,
 'TP': np.int64(104),
 'FN': np.int64(10),
 'TN': np.int64(4760),
 'FP': np.int64(1452)}

**Remark:**

In the stricter test dataset, I see a small improvement in sensitivity and negative predictive value, from 71.79\% to 90.35\%, 99.46\% to 99.79\% respectively. But it shows a lower precision. In terms of model performance, they are acceptably simiar.

**Conclusion:**

My selected features and logistic regression is insensitive to GCS imputation strategy both in the case of age <2 and age >= 2 group.
